In [15]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from pathlib import Path
import os
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.query import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *

In [17]:
query = Query("""SELECT Customer.Id as CustomerId, Name as Name,
                Location.Latitude as Latitude,
                Location.Longitude as Longitude,
                Location.Description as Location
                FROM Customer 
                LEFT JOIN Location ON Customer.Id = Location.CustomerId
                WHERE Active = 1""")
customer_eu1 = query.execute(EU1_Conn)
customer_eu1['DBLocation'] = 'EU1'
customer_eu2 = query.execute(EU2_Conn)
customer_eu2['DBLocation'] = 'EU2'
customer_df = pd.concat([customer_eu1, customer_eu2])

customer_df = customer_df[~customer_df['Name'].str.upper().str.contains('PICARRO', case=False) & ~customer_df['Name'].str.upper().str.contains('TEST', case=False)]
customer_df = customer_df[~customer_df['Name'].str.strip().str.upper().eq('ITALGAS-SERVICE')]


In [18]:
import reverse_geocoder as rg

ISO3166 = {
    "AD": "Andorra", "AL": "Albania", "AT": "Austria", "BA": "Bosnia and Herzegovina",
    "BE": "Belgium", "BG": "Bulgaria", "BY": "Belarus", "CH": "Switzerland",
    "CY": "Cyprus", "CZ": "Czechia", "DE": "Germany", "DK": "Denmark",
    "EE": "Estonia", "ES": "Spain", "FI": "Finland", "FR": "France",
    "GB": "United Kingdom", "GR": "Greece", "HR": "Croatia", "HU": "Hungary",
    "IE": "Ireland", "IS": "Iceland", "IT": "Italy", "LI": "Liechtenstein",
    "LT": "Lithuania", "LU": "Luxembourg", "LV": "Latvia", "MC": "Monaco",
    "MD": "Moldova", "ME": "Montenegro", "MK": "North Macedonia", "MT": "Malta",
    "NL": "Netherlands", "NO": "Norway", "PL": "Poland", "PT": "Portugal",
    "RO": "Romania", "RS": "Serbia", "RU": "Russia", "SE": "Sweden",
    "SI": "Slovenia", "SK": "Slovakia", "SM": "San Marino", "UA": "Ukraine",
    "VA": "Vatican City", "XK": "Kosovo",
}

located = customer_df.dropna(subset=["Latitude", "Longitude"]).copy()
if not located.empty:
    results = rg.search(
        list(zip(located["Latitude"].astype(float), located["Longitude"].astype(float))),
        mode=1,
    )
    located["Country"] = [ISO3166.get(r["cc"], r["cc"]) for r in results]
    country_by_customer = located.groupby("CustomerId")["Country"].agg(
        lambda s: s.mode().iloc[0] if not s.mode().empty else None
    )
else:
    country_by_customer = pd.Series(dtype=object)

customer_df = (
    customer_df.groupby(["CustomerId", "Name", "DBLocation"], as_index=False)
    .agg(
        Latitude=("Latitude", "mean"),
        Longitude=("Longitude", "mean"),
        LocationCount=("CustomerId", "size"),
    )
)
customer_df["Country"] = customer_df["CustomerId"].map(country_by_customer)

In [ ]:
customer_df = customer_df[customer_df['Country'] != 'US']
customer_df.loc[customer_df['Country'] == 'GH', 'Country'] = 'Italy'
country_list = customer_df["Country"].dropna().unique()
country_list = pd.DataFrame(country_list, columns=["Country"]).reset_index(drop=True)

,Country
0,United Kingdom
1,Italy
2,Romania
3,Switzerland
4,Greece
5,Germany
6,Poland
7,Austria
8,Ireland
9,Czechia


In [20]:
def make_sequential_country_id(idx):
    # Format: 00000000-0000-0000-0000-000000000001, incrementing last 12 digits
    return f"00000000-0000-0000-0000-{idx+1:012d}"

country_list.insert(0, "CountryId", [make_sequential_country_id(i) for i in range(len(country_list))])
country_list.rename(columns={'Country': 'Name'}, inplace=True)

In [22]:
KPI_Country.reinit_table(arguments = {'db_path': DB_PATH})
KPI_Country.query_table(arguments = {'db_path': DB_PATH})


,CountryId,Name,LastUpdated


In [23]:
KPI_Country.update_table(arguments = {'DataFrame': country_list, 'db_path': DB_PATH, 'PrimaryKey': 'CountryId'})

In [24]:
KPI_Country.query_table(arguments = {'db_path': DB_PATH})

,CountryId,Name,LastUpdated
0,00000000-0000-0000-0000-000000000001,United Kingdom,None
1,00000000-0000-0000-0000-000000000002,Italy,None
2,00000000-0000-0000-0000-000000000003,Romania,None
3,00000000-0000-0000-0000-000000000004,Switzerland,None
4,00000000-0000-0000-0000-000000000005,Greece,None
5,00000000-0000-0000-0000-000000000006,Germany,None
6,00000000-0000-0000-0000-000000000007,Poland,None
7,00000000-0000-0000-0000-000000000008,Austria,None
8,00000000-0000-0000-0000-000000000009,Ireland,None
9,00000000-0000-0000-0000-000000000010,Czechia,None


In [25]:
for _, row in customer_df.iterrows():
    customer_name = row["Name"]
    db_location = row["DBLocation"]
    country = row["Country"] if pd.notna(row["Country"]) else None
    conn = EU1_Conn if db_location == "EU1" else EU2_Conn
    add_customer(customer_name, conn, country=country)

In [26]:
print(DB_PATH)

database/KPIHub.db


In [27]:
KPI_Customer.query_table(arguments = {'db_path': DB_PATH})

,CustomerId,Name,ShortName,Active,DBLocation,Country,LastUpdated
0,CFFB9000-94BD-BA72-B352-39EBA962116D,ITALGAS,ITALGAS,1,EU2,Italy,2026-08-25 14:53:50.806890
1,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,Westnetz,Westnetz,1,EU1,Germany,2026-08-25 14:53:50.768058
2,B511B372-18D7-E520-1328-39F05C8E531A,Toscana Energia,ToscanaEnergia,1,EU2,Italy,2026-08-25 14:53:50.708351
3,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Wales and West Utilities,WalesandWestUtilities,1,EU1,United Kingdom,2026-08-25 14:53:49.844193
4,03F0CC2A-4D6B-00CE-755E-39EC4E48CDA8,APRETIGAS,APRETIGAS,1,EU2,Italy,2026-08-25 14:53:49.868503
...,...,...,...,...,...,...,...
74,EFDE940F-0468-EC5E-7465-3A22E9BBD13A,Multiservizi Azzanese,MultiserviziAzzanese,1,EU1,Italy,2026-08-25 14:53:50.971935
75,F41C2600-47D6-897B-C937-3A155A74DE88,AMAG Reti,AMAGReti,1,EU1,Italy,2026-08-25 14:53:50.981975
76,F49DD290-D6FD-482E-72A5-3A0E2D71A098,AMGAS Foggia,AMGASFoggia,1,EU2,Italy,2026-08-25 14:53:50.998047
77,F746CB4F-5E6A-FDAA-B67F-39F7D08BEA4A,RETI Distribuzione,RETIDistribuzione,1,EU2,Italy,2026-08-25 14:53:51.016352
